In [1]:
!PIP install yfinance

In [2]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist

In [7]:
# Download Data
ticker1 = yf.download('COKE', start='2020-01-01', end='2024-12-31')
ticker2 = yf.download('PEP', start='2020-01-01', end='2024-12-31')

# Extract and align closing prices
coke_close = ticker1['Close']
pep_close = ticker2['Close']

combined = pd.concat([coke_close, pep_close], axis=1).dropna()
combined.columns = ['COKE', 'PEP']

print(combined.shape)
print(combined.head())


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

(1257, 2)
                 COKE         PEP
Date                             
2020-01-02  26.822592  112.112396
2020-01-03  27.141727  111.955566
2020-01-06  26.785545  112.384781
2020-01-07  26.514851  110.618340
2020-01-08  26.334389  111.187889


In [15]:
# Compute local features for each series (Franses & Wiemann 2020, Equation 3)
def compute_local_features(series):
    values = series.values
    n = len(values)
    features = np.zeros((n-2, 2))

    for i in range(1, n-1):
        backward = (values[i] - values[i-1]) / values[i-1]
        forward = (values[i+1] - values[i]) / values[i]
        features[i-1] = [backward, forward]

    return features

coke_local = compute_local_features(combined['COKE'])
pep_local = compute_local_features(combined['PEP'])

print(f'Local feature shape - COKE: {coke_local.shape}, PEP: {pep_local.shape}')
print(f'First 3 COKE local features :\n{coke_local[:3]}')
print(f'First 3 PEP local features :\n{pep_local[:3]}')

Local feature shape - COKE: (1255, 2), PEP: (1255, 2)
First 3 COKE local features :
[[ 0.01189802 -0.01312304]
 [-0.01312304 -0.010106  ]
 [-0.010106   -0.00680607]]
First 3 PEP local features :
[[-0.00139886  0.00383379]
 [ 0.00383379 -0.0157178 ]
 [-0.0157178   0.00514878]]


In [18]:
# Compute global features for each series (Franses & Wiemann 2020, Equation 4)

def compute_global_features(series):
    values = series.values
    n = len(values)
    features = np.zeros((n-2,2))

    for i in range(1, n-1):
        past_avg = np.mean(values[:i])
        future_avg = np.mean(values[i+1:])

        past_dev = (values[i] - past_avg) / past_avg
        future_dev = (values[i] - future_avg) / future_avg

        features[i-1] = [past_dev, future_dev]

    return features

coke_global = compute_global_features(combined['COKE'])
pep_global = compute_global_features(combined['PEP'])

print(f'Global feature shape - COKE: {coke_global.shape}, PEP: {pep_global.shape}')
print(f'First 3 COKE global features :\n{coke_global[:3]}')
print(f'First 3 PEP global features :\n{pep_global[:3]}')

Global feature shape - COKE: (1255, 2), PEP: (1255, 2)
First 3 COKE global features :
[[ 0.01189802 -0.50613952]
 [-0.00728682 -0.51281963]
 [-0.0149265  -0.51794226]]
First 3 PEP global features :
[[-0.00139886 -0.21721007]
 [ 0.00313119 -0.21434323]
 [-0.01366529 -0.2268319 ]]


In [20]:
# Compute feature-based distance matrix (Franses & Wiemann 2020, Equation 5)

n = len(coke_local)
m = len(pep_local)

distance_matrix = np.zeros((n, m))

for i in range(n):
    for j in range(m):
        local_dist = (abs(coke_local[i, 0] - pep_local[j, 0]) + 
                      abs(coke_local[i, 1] - pep_local[j, 1]))

        global_dist = (abs(coke_global[i, 0] - pep_global[j,0]) +
                       abs(coke_global[i, 1] - pep_global[j,1]))

        distance_matrix[i,j] = local_dist + global_dist

print(f'Distance matrix shape: {distance_matrix.shape}')
print(f'Sample distances (first 3x3) :\n{distance_matrix[:3, :3]}')
                    
        

Distance matrix shape: (1255, 1255)
Sample distances (first 3x3) :
[[0.33248004 0.3112221  0.35075857]
 [0.32716149 0.33146305 0.31021573]
 [0.33360682 0.34450824 0.30993821]]


In [24]:
# Compute DTW warping path through the feature-based distance matrix
# Using dynamic programming to find the optimal path (Equation 1)

def dtw_from_distance_matrix(D):
    n, m = D.shape

    gamma = np.full((n,m), np.inf)
    gamma[0,0] = D[0,0]

    # fill first row
    for j in range(1,m):
        gamma[0,j] = D[0,j] + gamma[0, j-1]

    # fill the first column
    for i in range(1, n):
        gamma[0,i] = D[0,i] + gamma[0,i-1]

    # fill rest of matrix
    for i in range(1,n):
        for j in range(1,m):
            gamma[i,j] = D[i,j] + min(gamma[i-1,j-1],
                                      gamma[i-1, j],
                                      gamma[i,j-1])

    # Total DTW distance
    dtw_distance = gamma[n-1,m-1]

    # Normalized DTW distance (Equation 2)
    dtw_normalized = dtw_distance / (n + m)

    return dtw_distance, dtw_normalized, gamma

distance_raw, distance_norm, gamma = dtw_from_distance_matrix(distance_matrix)

print(f"Feature-based DTW distance (raw): {distance_raw:.4f}")
print(f"Feature-based DTW distance (normalized): {distance_norm:.4f}")
print(f"\nComparison:")
print(f"Z-score DTW:        18.1058  (normalized: {18.1058/2514:.4f})")
print(f"Log price DTW:      28.9630  (normalized: {28.9630/2514:.4f})")
print(f"Feature-based DTW:  {distance_raw:.4f}  (normalized: {distance_norm:.4f})")
    

Feature-based DTW distance (raw): 921.9977
Feature-based DTW distance (normalized): 0.3673

Comparison:
Z-score DTW:        18.1058  (normalized: 0.0072)
Log price DTW:      28.9630  (normalized: 0.0115)
Feature-based DTW:  921.9977  (normalized: 0.3673)


In [ ]:
# Visualize the feauture-base